# 03. Kullback-Leibler (KL) & Jensen-Shannon (JS) Divergence

**The mathematics of measuring statistical distance between probability distributions, with direct applications in Variational Autoencoders (VAEs), t-SNE, and GANs.**

---

## 1. What is Kullback-Leibler (KL) Divergence?

**KL Divergence** (also called **Relative Entropy**) measures how much information is lost when we approximate a true distribution $P(x)$ with a candidate distribution $Q(x)$:

$$D_{KL}(P \parallel Q) = \sum_x P(x) \log \left(\frac{P(x)}{Q(x)}\right) = \sum_x P(x) \log P(x) - \sum_x P(x) \log Q(x) = H(P, Q) - H(P)$$

### Key Properties:
1. **Non-Negativity (Gibbs' Inequality)**: $D_{KL}(P \parallel Q) \geq 0$, with equality $D_{KL} = 0 \iff P = Q$.
2. **Asymmetry (Not a true distance metric!)**:
   $$D_{KL}(P \parallel Q) \neq D_{KL}(Q \parallel P)$$
   - **Forward KL $D_{KL}(P \parallel Q)$ (Zero-avoiding / Mode-covering)**: Heavy penalty when $P(x) > 0$ but $Q(x) \approx 0$. Forces $Q$ to cover all modes of $P$.
   - **Reverse KL $D_{KL}(Q \parallel P)$ (Zero-forcing / Mode-seeking)**: Heavy penalty when $Q(x) > 0$ but $P(x) \approx 0$. Forces $Q$ to focus tightly on a single mode (used in Variational Inference).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def kl_divergence(p, q):
    eps = 1e-12
    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)
    return np.sum(p * np.log(p / q))

# Discrete distributions P and Q
P = np.array([0.4, 0.35, 0.25])
Q = np.array([0.333, 0.333, 0.334])

print("P:", P)
print("Q:", Q)
print(f"Forward KL  D_KL(P || Q): {kl_divergence(P, Q):.6f}")
print(f"Reverse KL  D_KL(Q || P): {kl_divergence(Q, P):.6f}")
print("Asymmetry demonstrated: D_KL(P || Q) != D_KL(Q || P)")


---

## 2. Analytical KL Divergence Between Two Gaussians (VAE Loss)

In **Variational Autoencoders (VAEs)**, the encoder outputs mean $\mu$ and variance $\sigma^2$ for a latent representation $q(z) = \mathcal{N}(\mu, \sigma^2)$, which is regularized against a standard normal prior $p(z) = \mathcal{N}(0, 1)$.

The analytical KL divergence between $q(z) = \mathcal{N}(\mu, \sigma^2)$ and $p(z) = \mathcal{N}(0, 1)$ is:

$$D_{KL}(\mathcal{N}(\mu, \sigma^2) \parallel \mathcal{N}(0, 1)) = -\frac{1}{2} \sum_{j=1}^d \left( 1 + \log(\sigma_j^2) - \mu_j^2 - \sigma_j^2 \right)$$

### Why this is a masterpiece:
No Monte Carlo sampling is needed! This closed-form loss term forces the latent space of the VAE to be smooth, continuous, and centered at zero.


In [ ]:
def vae_gaussian_kl_loss(mu, logvar):
    # Closed-form KL Divergence between N(mu, exp(logvar)) and N(0, I)
    # used in PyTorch VAE models.
    # -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    return -0.5 * np.sum(1 + logvar - mu**2 - np.exp(logvar))

mu = np.array([0.5, -0.2, 0.1])
logvar = np.array([0.1, -0.4, 0.05]) # log(sigma^2)

kl_loss = vae_gaussian_kl_loss(mu, logvar)
print(f"Latent Mean mu: {mu}")
print(f"Latent Log-Variance: {logvar}")
print(f"Analytical VAE KL Loss: {kl_loss:.4f}")


---

## 3. Jensen-Shannon (JS) Divergence

### Why JS Divergence?
Since KL divergence is asymmetric and can approach infinity, the **Jensen-Shannon Divergence** provides a **symmetrical, smoothed, and bounded** alternative:

$$D_{JS}(P \parallel Q) = \frac{1}{2} D_{KL}(P \parallel M) + \frac{1}{2} D_{KL}(Q \parallel M) \quad \text{where } M = \frac{1}{2}(P + Q)$$

- **Symmetric**: $D_{JS}(P \parallel Q) = D_{JS}(Q \parallel P)$
- **Bounded**: $0 \leq D_{JS}(P \parallel Q) \leq \log(2) \approx 0.693$ nats (or 1 bit).
- **Core of GANs**: The original Generative Adversarial Network (Goodfellow et al., 2014) objective implicitly minimizes the JS Divergence between real and generated distributions!


In [ ]:
def js_divergence(p, q):
    m = 0.5 * (p + q)
    return 0.5 * kl_divergence(p, m) + 0.5 * kl_divergence(q, m)

js_dist = js_divergence(P, Q)
js_dist_rev = js_divergence(Q, P)

print(f"JS Divergence D_JS(P || Q): {js_dist:.6f}")
print(f"JS Divergence D_JS(Q || P): {js_dist_rev:.6f}")
assert np.isclose(js_dist, js_dist_rev)
print("Symmetry verified!")


---

## 4. Summary & Key Takeaways

1. **KL Divergence** $D_{KL}(P \parallel Q) = \sum P \log(P/Q)$ measures statistical difference between probability distributions.
2. It is **asymmetric** ($D_{KL}(P \parallel Q) \neq D_{KL}(Q \parallel P)$) and non-negative.
3. The **analytical Gaussian KL divergence** is the exact regularization loss in Variational Autoencoders (VAEs).
4. **Jensen-Shannon (JS) Divergence** is a symmetric, bounded modification foundational to Generative Adversarial Networks (GANs).
